<a href="https://colab.research.google.com/github/animesh-kishore/llama3.2_3b_4bits_qlora/blob/main/llama3_2_3b_4bits_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

In [ ]:
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

# load quantized llama3.2 3B

# Set bnb config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # load llama3.2 3B in symmetric 4bits
    bnb_4bit_quant_type="nf4", # Quantization dtype
    bnb_4bit_use_double_quant=True, # Quantize scale as well. Saves more memory
    bnb_4bit_compute_dtype=torch.float16, # Quantized parameters are dequantized in this dtype before calculation
)

# Set tokenzier
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Used to make all sentences in a batch of same size. llama3.2 doesn't implicitly provide pad_token
tokenizer.padding_side = "right" # Add padding after end of sentences

# Set model
base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B",
    quantization_config=bnb_config,
    device_map="auto", # Automatically select underlying HW for execution
)
base_model.generation_config.pad_token_id = tokenizer.pad_token # llama3.2 doesn't have pad tokens implicitly set.

print(base_model)
print(f"Memory footprint {base_model.get_memory_footprint() / 1e6} MB")

In [ ]:
from peft import LoraConfig # peft is parameter efficient fine tuning

LORA_R = 32
LORA_ALPHA = 2 * LORA_R # Typically 2 * r
LORA_DROPOUT = 0.1 # zero out 10% of lora inputs on an average
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"] + ["gate_proj", "up_proj", "down_proj"] # Attention + MLP

# Create loRA config
lora_params = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none", # Do not update base model bias
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES
)

In [ ]:
from trl import SFTConfig # Transformer Reinforcement Learning used to refine LLMs post pre-training
from datetime import datetime

EPOCHS = 1
BATCH_SIZE = 32
OPTIMIZER = "paged_adamw_32bit"
MAX_SEQUENCE_LENGTH = 128
HUB_MODEL_ID="animeshkishore/llama3.2_3b_4bits_qlora"

# Create training config for Supervised Fine Tuning
train_params = SFTConfig(
    output_dir="sft_output", # local working directory
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, # per device (e.g. GPU, CPU) number of prompt-completion pairs that will be processed simultaneously
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1, # Accumulate gradient for multiple batches and update weights in one shot
    optim=OPTIMIZER,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.001,
    fp16=True, # Lora weights are still saved in fp32. However, during training it's casted to fp16 and used. Should be same as bnb_4bit_compute_dtype.
    max_grad_norm=0.3,
    warmup_ratio=0.01,
    group_by_length=True, # Group sequences of similar length in a batch together. Avoid memory wastage due to padding.
    max_length=MAX_SEQUENCE_LENGTH, # Max tokens per input sequence in a batch. Anything smaller is padded. Larger is chopped.
    save_steps=100, # saves checkpoint at output_dir
    save_total_limit=1, # Save only 1 checkpoint locally. Rest all will be saved on HF.
    save_strategy="steps",
    push_to_hub=True, # push checkpoints to HF as well
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="every_save", # push to HF for every local checkpoint save
    eval_strategy="steps",
    eval_steps=100, # Specify how often evaluation is run on validation dataset
    logging_steps=5, # Frequency at which training stats are send to wandb.ai
    report_to="wandb", # Send training stats to wandb.ai
    run_name=f"llama3.2_3b_4bits_qlora-{datetime.now():%Y-%m-%d_%H.%M.%S}", # Set run name to view on wandb.ai
)


In [ ]:
from trl import SFTTrainer
from datasets import load_dataset

DATASET_NAME = "ed-donner/items_prompts_lite"
dataset = load_dataset(DATASET_NAME) # load fine tuning dataset

# Create SFT trainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"].select(range(500)),
    peft_config=lora_params,
    args=train_params
)

In [ ]:
# Start training

fine_tuning.train()